# Qwen Image Edit Web Server (Rapid AIO) — Google Colab

Qwen-Image-Edit-Rapid-AIO-V23 (4-step accelerated) を Flask Web サーバーとして起動し、cloudflared で外部公開します。

**必要環境:** GPU ランタイム (T4 以上推奨)  
**特徴:** GGUF/nunchaku 不要。recent diffusers のみで動作。4ステップ高速推論。

## 1. GPU確認

In [ ]:
!nvidia-smi

## 2. パッケージインストール

In [ ]:
!pip install -q flask pillow huggingface_hub transformers accelerate safetensors googletrans
!pip install -q -U diffusers
!wget -qN https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

## 3. app_aio.py を配置

`server/app_aio.py` をアップロードするか、リポジトリからクローンしてください。

In [ ]:
import os, shutil

# --- 方法A: ファイルアップロード済みの場合 ---
# Google Colab のファイルブラウザ(左サイドバー)で app_aio.py をアップロードしてください

# --- 方法B: リポジトリからコピー (URLは適宜変更) ---
!git clone https://github.com/id-fa/simple-image-edit-with-qwen.git repo_tmp
!cp repo_tmp/server/app_aio.py .

os.makedirs("server", exist_ok=True)
if os.path.exists("app_aio.py") and not os.path.exists("server/app_aio.py"):
    shutil.copy("app_aio.py", "server/app_aio.py")

!ls -la server/app_aio.py 2>/dev/null || echo "⚠ ファイルが見つかりません。app_aio.py をアップロードしてください。"

## 4. 設定

パスワードやプリセットを変更できます。

In [ ]:
# --- 設定 ---
PASSWORD = "password"       # 生成パスワード
PORT = 5000                 # サーバーポート
GALLERY = True              # ギャラリーモード有効化
OFFLOAD = False             # True: sequential CPU offload (低VRAM向け、遅い)

# プリセット (空リストならボタン非表示)
PRESETS = [
    "高画質化::Enhance quality and fix artifacts.",
    "テキスト除去::Remove all overlaid text, subtitles, and credits.",
]

## 5. サーバー起動 + cloudflared 公開

Flask サーバーをバックグラウンドで起動し、cloudflared で公開URLを生成します。  
出力に表示される `https://******.trycloudflare.com` のURLにアクセスしてください。

In [ ]:
import subprocess, time, re, threading

SERVER_LOG = "server.log"

# コマンドライン引数を構築
cmd = [
    "python", "-u", "server/app_aio.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--password", PASSWORD,
]
if GALLERY:
    cmd.append("--gallery")
if OFFLOAD:
    cmd.append("--offload")
for preset in PRESETS:
    cmd.extend(["--preset", preset])

print(f"[colab] starting server: {' '.join(cmd)}")

# ログファイルに出力（パイプバッファ詰まり防止）
log_fh = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(cmd, stdout=log_fh, stderr=subprocess.STDOUT)

# ログを監視してサーバー起動完了を待機
print("[colab] waiting for model to load...")
ready = False
deadline = time.time() + 600  # 10分タイムアウト
seen = 0
while time.time() < deadline:
    time.sleep(2)
    if server_proc.poll() is not None:
        print("[colab] ERROR: server process exited unexpectedly")
        with open(SERVER_LOG) as f:
            print(f.read())
        break
    with open(SERVER_LOG) as f:
        lines = f.readlines()
    # 新しい行を表示
    for line in lines[seen:]:
        print(line, end="")
    seen = len(lines)
    if any("server starting at" in l for l in lines):
        ready = True
        break

if not ready:
    raise RuntimeError("Server failed to start. Check server.log")

print(f"\n[colab] server is running on port {PORT}")
print(f"[colab] starting cloudflared tunnel...")

# cloudflared 起動 (stderr にURL出力)
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)

# stderr を読み続けるスレッド（バッファ詰まり防止）
tunnel_lines = []
def drain_tunnel_stderr():
    for line in iter(tunnel_proc.stderr.readline, ""):
        tunnel_lines.append(line)
threading.Thread(target=drain_tunnel_stderr, daemon=True).start()

# URL抽出を待機
public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    time.sleep(1)
    for line in tunnel_lines:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            public_url = m.group(1)
            break
    if public_url:
        break

if public_url:
    print(f"\n{'='*60}")
    print(f"  Public URL: {public_url}")
    print(f"  Password:   {PASSWORD}")
    print(f"{'='*60}")
else:
    print("[colab] ERROR: cloudflared URL を取得できませんでした")
    print("tunnel log:", ''.join(tunnel_lines[-10:]))

## 6. ユーティリティ

ログ確認・サーバー停止用。

In [ ]:
# サーバーログ確認（直近20行）
!tail -20 server.log

In [ ]:
# サーバー + トンネル停止
server_proc.terminate()
tunnel_proc.terminate()
log_fh.close()
print("stopped")